Analysis of a new dataset: MoveAI
https://universe.roboflow.com/moval/moval/dataset/4

In [2]:
import os
import pandas as pd
#import plotly.express as px
import cv2 as cv
import argparse
import numpy as np
import time
import statistics
from datetime import datetime
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import sklearn.metrics



In [3]:
ruta_carpeta = r'C:\Users\inges\OneDrive - UDIT\src\photoneu\dataset\MovAl.v1i.tensorflow\train'

kernel_3 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(3,3))
kernel_5 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
RESIZE_FACTOR = 2.0
MAJOR_DEFECT_THRESHOLD = 12.0 / RESIZE_FACTOR #6.0 5.0 12.0
ELLIPSE_RATIO_THRESHOLD = 3.5 # 2.0
MIN_AREA = 300
MAX_AREA = 1200#2.5 * MIN_AREA
BW_THRES = 12 # min B/W value for threshold

normal_size = (480, 640)

# TriMouse
x_crop_min = int(normal_size[1]/19) #50
x_crop_max = int(normal_size[1]/25) # 30
y_crop_min = int(normal_size[0]/20) # 30
y_crop_max = int(normal_size[0]/20) # 16 -- 40

# 3mice dataset
#x_crop_min = int(normal_size[1]/7)#int(normal_size[1]/10) #50
#x_crop_max = int(normal_size[1]/7)#int(normal_size[1]/20) # 30
#y_crop_min = int(normal_size[0]/7)#int(normal_size[0]/16) # 30
#y_crop_max = int(normal_size[0]/5) #int(normal_size[0]/20) # 16 -- 40

SEGMENT_COLORS = [(0,255,0),(0,255,255),(255,255,0),(255,0,255)]


Lectura del fichero de anotaciones del dataset, con el formato: img_file_name, width, height, class, xmin, ymin, xmax, ymax 

In [8]:
# lectura del fichero de anotaciones
#formato del fichero filename, width, height, class, xmin, ymin, xmax, ymax (viene en la primera fila)

df_ground_truth = pd.read_csv(os.path.join(ruta_carpeta, '_annotations.csv'))

# calculamos el centroide de cada ratón y lo añadimos al dataframe con los campos mus_{i}_x, mus_{i}_y
df_centroids = pd.DataFrame()
for index, row in df_ground_truth.iterrows():
    img_file = row['filename']
    xmin = row['xmin']
    ymin = row['ymin']
    xmax = row['xmax']
    ymax = row['ymax']
    x_center = int((xmin + xmax) / 2)
    y_center = int((ymin + ymax) / 2)
    df_centroids = df_centroids.append({'filename': img_file, 'mus_x': x_center, 'mus_y': y_center}, ignore_index=True) 
df_centroids.head()

# reordenamos el dataframe para que tenga el formato img_file_name, width, height, num_mice, mus_{i}_x, mus_{i}_y,...
df_grouped = df_ground_truth.groupby('filename').agg({'width': 'first', 'height': 'first', 'class': 'count'}).rename(columns={'class': 'num_mice'}).reset_index()
df_grouped = df_grouped.merge(df_centroids, on='filename')
for i in range(1, int(df_grouped['num_mice'].max()) + 1):
    df_temp = df_ground_truth[df_ground_truth.groupby('filename').cumcount() == (i - 1)][['filename', 'xmin', 'ymin', 'xmax', 'ymax']]
    df_temp = df_temp.rename(columns={'xmin': f'mus_{i}_xmin', 'ymin': f'mus_{i}_ymin', 'xmax': f'mus_{i}_xmax', 'ymax': f'mus_{i}_ymax'})
    df_grouped = df_grouped.merge(df_temp, on='filename')

# borramos las columnas mus_x, mus_y
df_grouped = df_grouped.drop(columns=['mus_x', 'mus_y'])

# calculamos los centroides para cada ratón {i} y lo añadimos al dataframe como mus_{i}_x, mus_{i}_y
for i in range(1, int(df_grouped['num_mice'].max()) + 1):
    df_grouped[f'mus_{i}_x'] = ((df_grouped[f'mus_{i}_xmin'] + df_grouped[f'mus_{i}_xmax']) / 2).astype(int)
    df_grouped[f'mus_{i}_y'] = ((df_grouped[f'mus_{i}_ymin'] + df_grouped[f'mus_{i}_ymax']) / 2).astype(int)
df_grouped.rename(columns={"filename": "img_path"}, inplace=True)
# clear duplicates by img_path column
df_grouped.drop_duplicates(inplace=True)

# guardamos el dataframe en un fichero csv
df_grouped.to_csv(os.path.join(ruta_carpeta, 'ground_truth_centroids.csv'), index=False)


In [7]:
df_grouped.head()   


,img_path,width,height,num_mice,mus_1_xmin,mus_1_ymin,mus_1_xmax,mus_1_ymax,mus_2_xmin,mus_2_ymin,...,mus_3_xmin,mus_3_ymin,mus_3_xmax,mus_3_ymax,mus_1_x,mus_1_y,mus_2_x,mus_2_y,mus_3_x,mus_3_y
0,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,186,560,331,834,1171,173,...,1379,320,1639,491,258,697,1337,248,1509,405
1,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,186,560,331,834,1171,173,...,1379,320,1639,491,258,697,1337,248,1509,405
2,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,186,560,331,834,1171,173,...,1379,320,1639,491,258,697,1337,248,1509,405
3,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,1311,183,1629,314,1393,367,...,189,574,329,830,1470,248,1512,491,259,702
4,240712_3w_148b_151b_154b_m_rec_1000model_mp4-0...,1920,1080,3,1311,183,1629,314,1393,367,...,189,574,329,830,1470,248,1512,491,259,702
